In [2]:
!pip install segmentation-models-pytorch

In [3]:
!pip install lxml

In [5]:
# ============================================================
# FARMLAND BOUNDARY U-NET TRAINING
# ============================================================

import os
import cv2
import torch
import numpy as np
import xml.etree.ElementTree as ET

from glob import glob
from tqdm import tqdm
from PIL import Image, ImageDraw

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import segmentation_models_pytorch as smp

# ============================================================
# PATHS
# ============================================================

ANNOTATION_XML = r"D:\farmland boundary\New folder\Train\boundary mask\annotations.xml"

TRAIN_IMAGE_FOLDER = r"D:\farmland boundary\New folder\Train\farmlands"

PREDICT_IMAGE_FOLDER = r"D:\farmland boundary\New folder\images"

GENERATED_MASK_FOLDER = r"D:\farmland boundary\generated_masks"

PREDICTION_FOLDER = r"D:\farmland boundary\predicted_masks"

os.makedirs(GENERATED_MASK_FOLDER, exist_ok=True)
os.makedirs(PREDICTION_FOLDER, exist_ok=True)

# ============================================================
# IMAGE SIZE
# ============================================================

IMAGE_SIZE = 256

# ============================================================
# CREATE MASKS FROM CVAT XML
# ============================================================

print("\nCreating masks from XML annotations...")

tree = ET.parse(ANNOTATION_XML)
root = tree.getroot()

for image_tag in root.findall("image"):

    filename = image_tag.get("name")

    width = int(image_tag.get("width"))
    height = int(image_tag.get("height"))

    mask = Image.new("L", (width, height), 0)

    draw = ImageDraw.Draw(mask)

    for polygon in image_tag.findall("polygon"):

        points_str = polygon.get("points")

        points = []

        for p in points_str.split(";"):

            x, y = map(float, p.split(","))

            points.append((x, y))

        draw.polygon(points, outline=255, fill=0)

    mask_path = os.path.join(
        GENERATED_MASK_FOLDER,
        os.path.splitext(filename)[0] + ".png"
    )

    mask.save(mask_path)

print("Masks created successfully!")

# ============================================================
# DATASET
# ============================================================

class FarmlandDataset(Dataset):

    def __init__(self, image_folder, mask_folder):

        self.image_paths = sorted(
            glob(os.path.join(image_folder, "*.jpg")) +
            glob(os.path.join(image_folder, "*.png"))
        )

        self.mask_folder = mask_folder

        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):

    image_path = self.image_paths[idx]

    filename = os.path.basename(image_path)

    mask_path = os.path.join(
        self.mask_folder,
        os.path.splitext(filename)[0] + ".png"
    )

    image = cv2.imread(image_path)

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(mask_path, 0)

    # ========================================================
    # HANDLE MISSING MASKS
    # ========================================================

    if mask is None:

        print(f"Missing mask for: {filename}")

        # create empty mask
        mask = np.zeros(
            (image.shape[0], image.shape[1]),
            dtype=np.uint8
        )

    # ========================================================
    # RESIZE
    # ========================================================

    image = cv2.resize(
        image,
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    mask = cv2.resize(
        mask,
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    # ========================================================
    # NORMALIZE
    # ========================================================

    image = self.transform(image)

    mask = torch.tensor(
        mask / 255.0
    ).float().unsqueeze(0)

    return image, mask

# ============================================================
# LOAD DATASET
# ============================================================

dataset = FarmlandDataset(
    TRAIN_IMAGE_FOLDER,
    GENERATED_MASK_FOLDER
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

print(f"\nTraining images: {len(dataset)}")

# ============================================================
# MODEL
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

model.to(DEVICE)

# ============================================================
# LOSS + OPTIMIZER
# ============================================================

loss_fn = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# ============================================================
# TRAINING LOOP
# ============================================================

EPOCHS = 20

print("\nStarting training...\n")

for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0

    for images, masks in loader:

        images = images.to(DEVICE)

        masks = masks.to(DEVICE)

        outputs = model(images)

        loss = loss_fn(outputs, masks)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {epoch_loss:.4f}")

# ============================================================
# SAVE MODEL
# ============================================================

torch.save(
    model.state_dict(),
    r"D:\farmland boundary\farmland_unet.pth"
)

print("\nModel saved!")

# ============================================================
# PREDICTION ON 50 NEW IMAGES
# ============================================================

print("\nPredicting masks on unseen images...\n")

predict_paths = sorted(
    glob(os.path.join(PREDICT_IMAGE_FOLDER, "*.jpg")) +
    glob(os.path.join(PREDICT_IMAGE_FOLDER, "*.png"))
)[:50]

model.eval()

with torch.no_grad():

    for image_path in tqdm(predict_paths):

        image = cv2.imread(image_path)

        original_size = image.shape[:2]

        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        resized = cv2.resize(rgb, (IMAGE_SIZE, IMAGE_SIZE))

        tensor = transforms.ToTensor()(resized)

        tensor = tensor.unsqueeze(0).to(DEVICE)

        output = model(tensor)

        pred = torch.sigmoid(output)

        pred = pred.squeeze().cpu().numpy()

        pred = (pred > 0.5).astype(np.uint8) * 255

        pred = cv2.resize(
            pred,
            (original_size[1], original_size[0])
        )

        filename = os.path.basename(image_path)

        save_path = os.path.join(
            PREDICTION_FOLDER,
            os.path.splitext(filename)[0] + "_pred.png"
        )

        cv2.imwrite(save_path, pred)

print("\n===================================")
print("TRAINING + PREDICTION COMPLETE")
print("===================================")

IndentationError: expected an indented block after function definition on line 107 (376959593.py, line 109)

In [6]:
# ============================================================
# FARMLAND BOUNDARY U-NET TRAINING + PREDICTION
# FULL CORRECTED CODE
# ============================================================

import os
import cv2
import torch
import numpy as np
import xml.etree.ElementTree as ET

from glob import glob
from tqdm import tqdm
from PIL import Image, ImageDraw

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import segmentation_models_pytorch as smp

# ============================================================
# PATHS
# ============================================================

ANNOTATION_XML = r"D:\farmland boundary\New folder\Train\boundary mask\annotations.xml"

TRAIN_IMAGE_FOLDER = r"D:\farmland boundary\New folder\Train\farmlands"

PREDICT_IMAGE_FOLDER = r"D:\farmland boundary\New folder\images"

GENERATED_MASK_FOLDER = r"D:\farmland boundary\generated_masks"

PREDICTION_FOLDER = r"D:\farmland boundary\predicted_masks"

MODEL_SAVE_PATH = r"D:\farmland boundary\farmland_unet.pth"

os.makedirs(GENERATED_MASK_FOLDER, exist_ok=True)
os.makedirs(PREDICTION_FOLDER, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================

IMAGE_SIZE = 256
BATCH_SIZE = 2
EPOCHS = 20

# ============================================================
# DEVICE
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nUsing device: {DEVICE}")

# ============================================================
# STEP 1 — CREATE MASKS FROM CVAT XML
# ============================================================

print("\n===================================")
print("CREATING MASKS FROM XML")
print("===================================")

tree = ET.parse(ANNOTATION_XML)
root = tree.getroot()

created_masks = 0

for image_tag in root.findall("image"):

    filename = image_tag.get("name")

    width = int(float(image_tag.get("width")))
    height = int(float(image_tag.get("height")))

    # black background
    mask = Image.new("L", (width, height), 0)

    draw = ImageDraw.Draw(mask)

    polygons_found = False

    for polygon in image_tag.findall("polygon"):

        polygons_found = True

        points_str = polygon.get("points")

        points = []

        for p in points_str.split(";"):

            x, y = map(float, p.split(","))

            points.append((x, y))

        # draw white boundary
        draw.line(points + [points[0]], fill=255, width=2)

    if polygons_found:

        mask_filename = os.path.splitext(filename)[0] + ".png"

        mask_path = os.path.join(
            GENERATED_MASK_FOLDER,
            mask_filename
        )

        mask.save(mask_path)

        created_masks += 1

print(f"\nCreated {created_masks} masks")

# ============================================================
# CHECK GENERATED MASKS
# ============================================================

generated_masks = glob(
    os.path.join(GENERATED_MASK_FOLDER, "*.png")
)

print(f"Generated mask files: {len(generated_masks)}")

# ============================================================
# DATASET
# ============================================================

class FarmlandDataset(Dataset):

    def __init__(self, image_folder, mask_folder):

        self.image_paths = sorted(
            glob(os.path.join(image_folder, "*.jpg")) +
            glob(os.path.join(image_folder, "*.png")) +
            glob(os.path.join(image_folder, "*.jpeg"))
        )

        self.mask_folder = mask_folder

        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]

        filename = os.path.basename(image_path)

        mask_filename = os.path.splitext(filename)[0] + ".png"

        mask_path = os.path.join(
            self.mask_folder,
            mask_filename
        )

        # ====================================================
        # LOAD IMAGE
        # ====================================================

        image = cv2.imread(image_path)

        if image is None:
            raise ValueError(f"Could not read image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # ====================================================
        # LOAD MASK
        # ====================================================

        mask = cv2.imread(mask_path, 0)

        # if mask missing create empty mask
        if mask is None:

            print(f"Missing mask for: {filename}")

            mask = np.zeros(
                (image.shape[0], image.shape[1]),
                dtype=np.uint8
            )

        # ====================================================
        # RESIZE
        # ====================================================

        image = cv2.resize(
            image,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        mask = cv2.resize(
            mask,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        # ====================================================
        # NORMALIZE
        # ====================================================

        image = self.transform(image)

        mask = torch.tensor(
            mask / 255.0
        ).float().unsqueeze(0)

        return image, mask

# ============================================================
# LOAD DATASET
# ============================================================

dataset = FarmlandDataset(
    TRAIN_IMAGE_FOLDER,
    GENERATED_MASK_FOLDER
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f"\nTraining images found: {len(dataset)}")

# ============================================================
# MODEL
# ============================================================

print("\n===================================")
print("LOADING U-NET MODEL")
print("===================================")

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

model.to(DEVICE)

# ============================================================
# LOSS FUNCTION
# ============================================================

loss_fn = torch.nn.BCEWithLogitsLoss()

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

# ============================================================
# TRAINING LOOP
# ============================================================

print("\n===================================")
print("STARTING TRAINING")
print("===================================\n")

for epoch in range(EPOCHS):

    model.train()

    epoch_loss = 0

    for images, masks in loader:

        images = images.to(DEVICE)

        masks = masks.to(DEVICE)

        # forward
        outputs = model(images)

        # loss
        loss = loss_fn(outputs, masks)

        # backward
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {epoch_loss:.4f}"
    )

# ============================================================
# SAVE MODEL
# ============================================================

torch.save(
    model.state_dict(),
    MODEL_SAVE_PATH
)

print("\n===================================")
print("MODEL SAVED")
print("===================================")

# ============================================================
# STEP 2 — PREDICT ON 50 UNSEEN IMAGES
# ============================================================

print("\n===================================")
print("PREDICTING ON NEW IMAGES")
print("===================================\n")

predict_paths = sorted(
    glob(os.path.join(PREDICT_IMAGE_FOLDER, "*.jpg")) +
    glob(os.path.join(PREDICT_IMAGE_FOLDER, "*.png")) +
    glob(os.path.join(PREDICT_IMAGE_FOLDER, "*.jpeg"))
)[:50]

print(f"Predicting on {len(predict_paths)} images")

model.eval()

with torch.no_grad():

    for image_path in tqdm(predict_paths):

        image = cv2.imread(image_path)

        if image is None:
            continue

        original_h, original_w = image.shape[:2]

        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        resized = cv2.resize(
            rgb,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        tensor = transforms.ToTensor()(resized)

        tensor = tensor.unsqueeze(0).to(DEVICE)

        # prediction
        output = model(tensor)

        pred = torch.sigmoid(output)

        pred = pred.squeeze().cpu().numpy()

        # threshold
        pred = (pred > 0.5).astype(np.uint8) * 255

        # resize back
        pred = cv2.resize(
            pred,
            (original_w, original_h)
        )

        # save
        filename = os.path.basename(image_path)

        save_path = os.path.join(
            PREDICTION_FOLDER,
            os.path.splitext(filename)[0] + "_pred.png"
        )

        cv2.imwrite(save_path, pred)

print("\n===================================")
print("TRAINING + PREDICTION COMPLETE")
print("===================================")

print(f"\nPredicted masks saved in:")
print(PREDICTION_FOLDER)


Using device: cpu

CREATING MASKS FROM XML

Created 10 masks
Generated mask files: 10

Training images found: 10

LOADING U-NET MODEL

STARTING TRAINING

Missing mask for: 125 - Copy.jpg
Missing mask for: 150 - Copy.jpg
Missing mask for: 113 - Copy.jpg
Missing mask for: 149 - Copy.jpg
Epoch 1/20 Loss: 1.9109
Missing mask for: 149 - Copy.jpg
Missing mask for: 125 - Copy.jpg
Missing mask for: 113 - Copy.jpg
Missing mask for: 150 - Copy.jpg
Epoch 2/20 Loss: 1.8178
Missing mask for: 149 - Copy.jpg
Missing mask for: 150 - Copy.jpg
Missing mask for: 125 - Copy.jpg
Missing mask for: 113 - Copy.jpg
Epoch 3/20 Loss: 1.7530
Missing mask for: 125 - Copy.jpg
Missing mask for: 149 - Copy.jpg
Missing mask for: 113 - Copy.jpg
Missing mask for: 150 - Copy.jpg
Epoch 4/20 Loss: 1.6674
Missing mask for: 125 - Copy.jpg
Missing mask for: 149 - Copy.jpg
Missing mask for: 113 - Copy.jpg
Missing mask for: 150 - Copy.jpg
Epoch 5/20 Loss: 1.6057
Missing mask for: 125 - Copy.jpg
Missing mask for: 150 - Copy.jpg

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:07<00:00,  6.60it/s]


TRAINING + PREDICTION COMPLETE

Predicted masks saved in:
D:\farmland boundary\predicted_masks


In [7]:
import os
import cv2
import numpy as np
import xml.etree.ElementTree as ET

# ============================================================
# PATHS
# ============================================================

XML_PATH = r"D:\farmland boundary\New folder\Train\boundary mask\annotations.xml"

OUTPUT_MASKS = r"D:\farmland boundary\boundary_masks"

os.makedirs(OUTPUT_MASKS, exist_ok=True)

# ============================================================
# LOAD XML
# ============================================================

tree = ET.parse(XML_PATH)
root = tree.getroot()

# ============================================================
# CREATE MASKS
# ============================================================

for image_tag in root.findall("image"):

    filename = image_tag.attrib["name"]

    width = int(image_tag.attrib["width"])
    height = int(image_tag.attrib["height"])

    # BLACK MASK
    mask = np.zeros((height, width), dtype=np.uint8)

    # ========================================================
    # DRAW ONLY BOUNDARIES
    # ========================================================

    for poly in image_tag.findall("polygon"):

        label = poly.attrib.get("label", "")

        # ONLY FARMLAND
        if label != "farmland":
            continue

        points_str = poly.attrib["points"]

        points = []

        for p in points_str.split(";"):

            x, y = map(float, p.split(","))

            points.append([int(x), int(y)])

        points = np.array(points, dtype=np.int32)

        # ====================================================
        # DRAW THIN WHITE BOUNDARY
        # ====================================================

        cv2.polylines(
            mask,
            [points],
            isClosed=True,
            color=255,
            thickness=2
        )

    # ========================================================
    # SAVE
    # ========================================================

    save_path = os.path.join(
        OUTPUT_MASKS,
        os.path.splitext(filename)[0] + ".png"
    )

    cv2.imwrite(save_path, mask)

    print("Saved:", save_path)

print("\nDONE")

Saved: D:\farmland boundary\boundary_masks\100 - Copy.png
Saved: D:\farmland boundary\boundary_masks\101 - Copy.png
Saved: D:\farmland boundary\boundary_masks\113.png
Saved: D:\farmland boundary\boundary_masks\125.png
Saved: D:\farmland boundary\boundary_masks\149.png
Saved: D:\farmland boundary\boundary_masks\150.png
Saved: D:\farmland boundary\boundary_masks\199 - Copy.png
Saved: D:\farmland boundary\boundary_masks\200 - Copy.png
Saved: D:\farmland boundary\boundary_masks\201 - Copy.png
Saved: D:\farmland boundary\boundary_masks\202 - Copy.png

DONE


In [8]:
# ============================================================
# IMPORTS
# ============================================================

import os
import cv2
import torch
import random
import numpy as np

from glob import glob
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import segmentation_models_pytorch as smp

# ============================================================
# PATHS
# ============================================================

TRAIN_IMAGES = r"D:\farmland boundary\New folder\Train\farmlands"

TRAIN_MASKS = r"D:\farmland boundary\boundary_masks"

PREDICT_IMAGES = r"D:\farmland boundary\New folder\images"

OUTPUT_PREDICTIONS = r"D:\farmland boundary\predicted_masks"

os.makedirs(OUTPUT_PREDICTIONS, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================

IMAGE_SIZE = 512

BATCH_SIZE = 2

EPOCHS = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# ============================================================
# DATASET
# ============================================================

class FarmlandDataset(Dataset):

    def __init__(self, image_folder, mask_folder):

        self.samples = []

        image_paths = sorted(
            glob(os.path.join(image_folder, "*.jpg")) +
            glob(os.path.join(image_folder, "*.png"))
        )

        for image_path in image_paths:

            filename = os.path.basename(image_path)

            mask_name = os.path.splitext(filename)[0] + ".png"

            mask_path = os.path.join(mask_folder, mask_name)

            if os.path.exists(mask_path):

                self.samples.append(
                    (image_path, mask_path)
                )

        print(f"Valid pairs: {len(self.samples)}")

        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        image_path, mask_path = self.samples[idx]

        # IMAGE
        image = cv2.imread(image_path)

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # MASK
        mask = cv2.imread(mask_path, 0)

        # RESIZE
        image = cv2.resize(
            image,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        mask = cv2.resize(
            mask,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        # NORMALIZE
        image = self.transform(image)

        mask = torch.tensor(
            mask / 255.0
        ).float().unsqueeze(0)

        return image, mask

# ============================================================
# LOAD DATASET
# ============================================================

dataset = FarmlandDataset(
    TRAIN_IMAGES,
    TRAIN_MASKS
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# ============================================================
# MODEL
# ============================================================

model = smp.Unet(

    encoder_name="resnet34",

    encoder_weights="imagenet",

    in_channels=3,

    classes=1
)

model = model.to(DEVICE)

# ============================================================
# LOSS + OPTIMIZER
# ============================================================

loss_fn = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# ============================================================
# TRAINING
# ============================================================

print("\nSTART TRAINING\n")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for images, masks in loader:

        images = images.to(DEVICE)

        masks = masks.to(DEVICE)

        preds = model(images)

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss: {total_loss:.4f}"
    )

# ============================================================
# SAVE MODEL
# ============================================================

MODEL_PATH = r"D:\farmland boundary\farmland_model.pth"

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print("\nMODEL SAVED")

# ============================================================
# PREDICTION ON 50 IMAGES
# ============================================================

print("\nPREDICTING...\n")

model.eval()

predict_images = sorted(
    glob(os.path.join(PREDICT_IMAGES, "*.jpg")) +
    glob(os.path.join(PREDICT_IMAGES, "*.png"))
)

# RANDOM 50
predict_images = random.sample(
    predict_images,
    min(50, len(predict_images))
)

transform = transforms.Compose([
    transforms.ToTensor()
])

with torch.no_grad():

    for image_path in tqdm(predict_images):

        image = cv2.imread(image_path)

        original_h, original_w = image.shape[:2]

        rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        resized = cv2.resize(
            rgb,
            (IMAGE_SIZE, IMAGE_SIZE)
        )

        tensor = transform(resized)

        tensor = tensor.unsqueeze(0).to(DEVICE)

        # PREDICT
        pred = model(tensor)

        pred = torch.sigmoid(pred)

        pred = pred.squeeze().cpu().numpy()

        # ====================================================
        # THRESHOLD
        # ====================================================

        pred = (pred > 0.2).astype(np.uint8)

        pred = pred * 255

        # ====================================================
        # CLEANING
        # ====================================================

        kernel = np.ones((2,2), np.uint8)

        pred = cv2.morphologyEx(
            pred,
            cv2.MORPH_OPEN,
            kernel
        )

        # ====================================================
        # RESIZE BACK
        # ====================================================

        pred = cv2.resize(
            pred,
            (original_w, original_h)
        )

        # ====================================================
        # SAVE
        # ====================================================

        filename = os.path.basename(image_path)

        save_path = os.path.join(
            OUTPUT_PREDICTIONS,
            os.path.splitext(filename)[0] + "_pred.png"
        )

        cv2.imwrite(save_path, pred)

print("\nDONE")

Using device: cpu
Valid pairs: 6

START TRAINING

Epoch 1/100 Loss: 1.7487
Epoch 2/100 Loss: 1.4143
Epoch 3/100 Loss: 1.1496
Epoch 4/100 Loss: 0.9765
Epoch 5/100 Loss: 0.8792
Epoch 6/100 Loss: 0.8046
Epoch 7/100 Loss: 0.7510
Epoch 8/100 Loss: 0.7000
Epoch 9/100 Loss: 0.6657
Epoch 10/100 Loss: 0.6393
Epoch 11/100 Loss: 0.6093
Epoch 12/100 Loss: 0.5785
Epoch 13/100 Loss: 0.5656
Epoch 14/100 Loss: 0.5516
Epoch 15/100 Loss: 0.5331
Epoch 16/100 Loss: 0.5177
Epoch 17/100 Loss: 0.5079
Epoch 18/100 Loss: 0.4953
Epoch 19/100 Loss: 0.4859
Epoch 20/100 Loss: 0.4720
Epoch 21/100 Loss: 0.4690
Epoch 22/100 Loss: 0.4674
Epoch 23/100 Loss: 0.4577
Epoch 24/100 Loss: 0.4413
Epoch 25/100 Loss: 0.4409
Epoch 26/100 Loss: 0.4326
Epoch 27/100 Loss: 0.4281
Epoch 28/100 Loss: 0.4172
Epoch 29/100 Loss: 0.4069
Epoch 30/100 Loss: 0.4034
Epoch 31/100 Loss: 0.4035
Epoch 32/100 Loss: 0.3995
Epoch 33/100 Loss: 0.3960
Epoch 34/100 Loss: 0.3893
Epoch 35/100 Loss: 0.3829
Epoch 36/100 Loss: 0.3970
Epoch 37/100 Loss: 0.40

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:21<00:00,  2.31it/s]


DONE
